# Eclipse Analysis Pipeline
**Self-contained — no external files needed beyond your image folder.**

### Setup (do this once)
1. Install VS Code: https://code.visualstudio.com
2. Install Python: https://python.org/downloads
3. Open a terminal in VS Code and run:
```
pip install opencv-python numpy pandas openpyxl
```
4. Open this notebook in VS Code and select your Python interpreter.

### Running
1. In **Cell 1**, set `SRC` to the folder containing your eclipse JPEGs.
2. Set `OUT` to wherever you want the results saved.
3. Run Cell 1 — that's it. The xlsx saves automatically when it finishes.

In [ ]:
import os, re, cv2
import numpy as np
import pandas as pd
from datetime import datetime

# ── CONFIGURE THESE TWO PATHS ───────────────────────────────
# SRC = folder containing your eclipse JPEGs
# OUT = folder where eclipse_measurements.xlsx will be saved
#
# Windows example:  r"C:\Users\Matth\Downloads\content"
# Mac/Linux example: "/Users/yourname/Downloads/content"

SRC = r"./images"  # <-- point this at your folder of eclipse JPEGs
OUT = r"./output"  # <-- results will be saved here

# ════════════════════════════════════════════════════════════
#  PIPELINE FUNCTIONS  (fully inlined — no imports needed)
# ════════════════════════════════════════════════════════════

TS_PAT = re.compile(r"(\d{4})_(\d{2})_(\d{2})_(\d{2})_(\d{2})_(\d{2})_(\d{3})")
RMIN, RMAX = 240, 330
R0_FALLBACK = 285.0


def parse_ts(fname):
    m = TS_PAT.search(fname)
    if not m:
        return None
    y, mo, d, h, mi, s, ms = map(int, m.groups())
    return datetime(y, mo, d, h, mi, s, ms * 1000)


def load_gray(path):
    img = cv2.imread(path)
    return img, cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)


def segment_sun(gray):
    h, w = gray.shape
    gb = cv2.GaussianBlur(gray, (5, 5), 0)
    otsu, _ = cv2.threshold(gb, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    thr = max(otsu, 18)
    bw = (gb > thr).astype(np.uint8) * 255
    bw[int(h * 0.92):, :] = 0
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    bw = cv2.morphologyEx(bw, cv2.MORPH_OPEN, k, 1)
    bw = cv2.morphologyEx(bw, cv2.MORPH_CLOSE, k, 2)
    n, lab, stats, _ = cv2.connectedComponentsWithStats(bw, 8)
    if n <= 1:
        return np.zeros_like(bw), 0

    def is_fov_ring(i):
        return (stats[i, cv2.CC_STAT_WIDTH] > 0.85 * w and
                stats[i, cv2.CC_STAT_HEIGHT] > 0.85 * h)

    cands = [i for i in range(1, n)
             if stats[i, cv2.CC_STAT_AREA] >= 50
             and not is_fov_ring(i)
             and gb[lab == i].mean() >= 45]
    if not cands:
        cands = [i for i in range(1, n)
                 if stats[i, cv2.CC_STAT_AREA] >= 50 and not is_fov_ring(i)]
    if not cands:
        cands = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))]
    best_i = max(cands, key=lambda i: stats[i, cv2.CC_STAT_AREA])
    mask = np.where(lab == best_i, 255, 0).astype(np.uint8)
    return mask, int(stats[best_i, cv2.CC_STAT_AREA])


def _taubin(pts):
    x = pts[:, 0].astype(np.float64); y = pts[:, 1].astype(np.float64)
    xm, ym = x.mean(), y.mean(); u, v = x - xm, y - ym
    Suu, Svv, Suv = (u*u).sum(), (v*v).sum(), (u*v).sum()
    Suuu, Svvv = (u**3).sum(), (v**3).sum()
    Suvv, Svuu = (u*v*v).sum(), (v*u*u).sum()
    A = np.array([[Suu, Suv], [Suv, Svv]])
    b = 0.5 * np.array([Suuu + Suvv, Svvv + Svuu])
    try:
        uc, vc = np.linalg.solve(A, b)
    except np.linalg.LinAlgError:
        return None
    cx, cy = uc + xm, vc + ym
    r = np.sqrt(uc*uc + vc*vc + (Suu + Svv) / len(x))
    return float(cx), float(cy), float(r)


def fit_solar_circle(mask, iters=1500, seed=0):
    cnts, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    if not cnts:
        return None
    c = max(cnts, key=cv2.contourArea).reshape(-1, 2).astype(np.float64)
    if len(c) < 8:
        return None
    rng = np.random.default_rng(seed)
    best, bestn = None, -1
    for _ in range(iters):
        f = _taubin(c[rng.choice(len(c), 3, replace=False)])
        if f is None:
            continue
        cx, cy, r = f
        if not (RMIN <= r <= RMAX):
            continue
        d = np.abs(np.hypot(c[:, 0]-cx, c[:, 1]-cy) - r)
        ni = int((d < 2.5).sum())
        if ni > bestn:
            bestn, best = ni, (cx, cy, r, d < 2.5)
    if best is None:
        return None
    cx, cy, r, inl = best
    f = _taubin(c[inl])
    if f:
        cx, cy, r = f
    d = np.abs(np.hypot(c[:, 0]-cx, c[:, 1]-cy) - r)
    rms = float(np.sqrt((d[d < 3]**2).mean())) if (d < 3).any() else 99.0
    reliable = (bestn >= 120 and rms < 2.2 and RMIN <= r <= RMAX)
    return dict(cx=cx, cy=cy, r=r, inliers=bestn, n=len(c), rms=rms, reliable=reliable)


def coverage_pct(gray, mask, circ, r0=R0_FALLBACK):
    if circ is not None and circ["reliable"]:
        cx, cy, R = circ["cx"], circ["cy"], circ["r"]
        disk = np.zeros(gray.shape, np.uint8)
        cv2.circle(disk, (int(round(cx)), int(round(cy))), int(round(R)), 255, -1)
        lit_inside = int(((mask > 0) & (disk > 0)).sum())
        a_full = np.pi * R * R
        cov = float(np.clip((1.0 - lit_inside / a_full) * 100.0, 0.0, 100.0))
        return cov, float(a_full), R, "high"
    R = r0
    a_full = np.pi * R * R
    area_lit = int((mask > 0).sum())
    cov = float(np.clip((1.0 - area_lit / a_full) * 100.0, 0.0, 100.0))
    return cov, float(a_full), R, "low"


def detect_sunspots(gray, mask, circ, r0=R0_FALLBACK):
    if circ is None:
        return 0, []
    cx, cy = circ["cx"], circ["cy"]
    r = circ["r"] if circ["reliable"] else r0
    inner = np.zeros_like(mask)
    cv2.circle(inner, (int(round(cx)), int(round(cy))), int(r * 0.93), 255, -1)
    region = cv2.bitwise_and(inner, mask)
    if int((region > 0).sum()) < 0.55 * np.pi * r * r:
        return 0, []
    g = gray.astype(np.float32)
    bg = cv2.GaussianBlur(g, (0, 0), sigmaX=max(5, r * 0.10))
    flat = bg - g
    flat[region == 0] = 0
    vals = flat[region > 0]
    thr = vals.mean() + 1.8 * vals.std()
    spot = ((flat > thr) & (region > 0)).astype(np.uint8) * 255
    spot = cv2.morphologyEx(spot, cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3)))
    cnts, _ = cv2.findContours(spot, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    spots, amin, amax = [], max(4, (r*0.010)**2*np.pi), (r*0.16)**2*np.pi
    for cc in cnts:
        a = cv2.contourArea(cc)
        if amin <= a <= amax:
            M = cv2.moments(cc)
            if M["m00"] > 0:
                sx, sy = M["m10"]/M["m00"], M["m01"]/M["m00"]
                if np.hypot(sx-cx, sy-cy) < 0.86 * r:
                    spots.append((float(sx), float(sy), float(a)))
    return len(spots), spots


def relative_luminosity(gray, mask):
    m = mask > 0
    return float(gray[m].astype(np.float64).sum()) if m.any() else 0.0


def crescent_features(mask, circ, r0=R0_FALLBACK):
    m = mask > 0
    ys, xs = np.where(m)
    if len(xs) == 0 or circ is None:
        return 0.0, 0.0
    R = circ["r"] if circ["reliable"] else r0
    off = np.hypot(xs.mean()-circ["cx"], ys.mean()-circ["cy"]) / max(R, 1)
    M = cv2.moments(mask, binaryImage=True)
    ang = 0.5*np.degrees(np.arctan2(2*M["mu11"], (M["mu20"]-M["mu02"]))) \
        if (M["mu20"]-M["mu02"]) or M["mu11"] else 0.0
    return float(ang), float(off)


def phase_label(c):
    if c < 10:  return "Full"
    if c < 50:  return "Large crescent"
    if c < 85:  return "Moderate crescent"
    return "Thin crescent"


# ════════════════════════════════════════════════════════════
#  MAIN: load images -> process -> save xlsx
# ════════════════════════════════════════════════════════════

files = sorted([f for f in os.listdir(SRC) if f.lower().endswith((".jpg", ".jpeg"))])
print(f"Found {len(files)} images in {SRC}")
if not files:
    raise FileNotFoundError(f"No JPEGs found in {SRC} — check your SRC path above.")

# Pass 1: segment + fit, collect reliable radii for calibration
recs = []
for i, f in enumerate(files):
    img, gray = load_gray(os.path.join(SRC, f))
    mask, area = segment_sun(gray)
    circ = fit_solar_circle(mask)
    recs.append(dict(file=f, ts=parse_ts(f), gray=gray, mask=mask,
                     circ=circ, area_lit_raw=area))
    if (i + 1) % 10 == 0:
        print(f"  Pass 1: {i+1}/{len(files)}")

rel_r = [r["circ"]["r"] for r in recs if r["circ"] and r["circ"]["reliable"]]
R0 = float(np.median(rel_r)) if rel_r else R0_FALLBACK
print(f"\nCalibrated R0 = {round(R0, 1)} px  ({len(rel_r)}/{len(files)} reliable fits)")

# Pass 2: coverage, sunspots, luminosity, features
rows = []
for i, r in enumerate(recs):
    cov, a_full, R, conf = coverage_pct(r["gray"], r["mask"], r["circ"], r0=R0)
    nsp, spots           = detect_sunspots(r["gray"], r["mask"], r["circ"], r0=R0)
    lum                  = relative_luminosity(r["gray"], r["mask"])
    ang, off             = crescent_features(r["mask"], r["circ"], r0=R0)
    rows.append(dict(
        File               = r["file"],
        Timestamp          = r["ts"].strftime("%Y-%m-%d %H:%M:%S.%f")[:-3] if r["ts"] else "",
        Day                = r["ts"].day if r["ts"] else None,
        Coverage_pct       = round(cov, 2),
        Fit_radius_px      = round(r["circ"]["r"], 1) if r["circ"] else None,
        Radius_used_px     = round(R, 1),
        Fit_RMS_px         = round(r["circ"]["rms"], 2) if r["circ"] else None,
        Fit_confidence     = conf,
        Sunspots           = nsp,
        Rel_luminosity     = round(lum, 1),
        Crescent_angle_deg = round(ang, 1),
        Centroid_offset    = round(off, 3),
    ))
    if (i + 1) % 10 == 0:
        print(f"  Pass 2: {i+1}/{len(recs)}")

df = pd.DataFrame(rows)

# Normalize luminosity against full-disk median
full_lum = df.loc[df["Coverage_pct"] < 5, "Rel_luminosity"]
lum_ref  = full_lum.median() if len(full_lum) else df["Rel_luminosity"].max()
df["Luminosity_norm"] = (df["Rel_luminosity"] / lum_ref).round(3)

# Phase labels
df["Phase_label"] = df["Coverage_pct"].apply(phase_label)

# Sort by timestamp
df["_sort"] = df["Timestamp"].replace("", "9999")
df = df.sort_values("_sort").drop(columns="_sort").reset_index(drop=True)

# Save xlsx
os.makedirs(OUT, exist_ok=True)
out_path = os.path.join(OUT, "eclipse_measurements.xlsx")
df.to_excel(out_path, index=False)
print(f"\nDone. Saved to: {out_path}")

# Summary
print(f"\nProcessed {len(df)} images")
print("\nPhase breakdown:")
print(df["Phase_label"].value_counts().to_string())
print("\nFit confidence breakdown:")
print(df["Fit_confidence"].value_counts().to_string())
print("\nFull-disk frames (coverage < 5%):")
print(df.loc[df["Coverage_pct"] < 5,
      ["Timestamp", "Coverage_pct", "Sunspots"]].to_string(index=False))
print("\nTop sunspot frames:")
print(df.nlargest(6, "Sunspots")[["Timestamp", "Coverage_pct", "Sunspots"]].to_string(index=False))